# Load Library

In [51]:
import json
import re
from collections import Counter

In [52]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

# The function used to clean the results.

In [ ]:
def clean_truefalse(response):
    # Find all exact occurrences of the words "Đúng" (True) or "Sai" (False), case-insensitive
    matches = re.findall(r'\b(Đúng|Sai)\b', response, re.IGNORECASE)
    matches = [m.capitalize() for m in matches]
    matches_set = set(matches)

    # If both "Đúng" and "Sai" are present → conflict → return empty string
    if 'Đúng' in matches_set and 'Sai' in matches_set:
        response = ''
    elif matches:
        # Keep only the first valid occurrence
        response = matches[0]
    else:
        # No valid match found
        response = ''

    return response

In [ ]:
def clean_multichoice(response):
    # Search for the first single letter match from A to D (case-insensitive), as a standalone word
    match = re.search(r'\b([A-D])\b', response, re.IGNORECASE)
    if match:
        # If a valid match is found, return it in uppercase
        response = match.group(1).upper()
    else:
        # No valid match found
        response = ''

    return response

# Load Result from file result_for_ensemble


In [55]:
grpo_qwen_fewshot = load_json("GRPO-VI-Qwen2-7B-RAG_private_test_task2.json")
qwen2 = load_json("Qwen2.5-7B-Instruct_private.json")
grpoviqwen = load_json("GRPO-VI-Qwen2-7B-RAG_private.json")
viqwen2 = load_json("Vi-Qwen2-7B-RAG_private.json")

In [ ]:
result_list = [grpoviqwen, qwen2, viqwen2, grpo_qwen_fewshot]

In [ ]:
def ensemble(TF_idx, MC_idx, FT_idx, output_file):
    ensemble_result = []

    # Iterate through all questions in the private test set
    for i in range(len(grpoviqwen)):

        item = grpoviqwen[i]
        responses = []

        # Handle True/False questions
        if item['question_type'] == "Đúng/Sai": 
            if TF_idx != 3: # Special case handling (hardcoded index check)
                final_answer = clean_truefalse(result_list[TF_idx][i]["predicted_answer"])
            else:
                final_answer = result_list[TF_idx][i]["answer"]
        elif item['question_type'] == "Trắc nghiệm":
            for j in MC_idx:
                # Clean and collect responses from all models
                cleaned_response = clean_multichoice(result_list[j][i]["predicted_answer"])
                responses.append(cleaned_response)
        else:
            final_answer = result_list[FT_idx][i]["answer"]

         # Majority voting for Multiple Choice questions
        if item['question_type'] == "Trắc nghiệm":
            most_common = Counter(responses).most_common(1)[0]
            if most_common[1] >= 2:
                # Use the most common answer if it appears at least twice
                final_answer = most_common[0]
            else:
                # Tie-breaker: use the first model's response (priority fallback)
                final_answer = responses[0]

        # Store the final result
        temp = {}
        temp['question_id'] = item['question_id']
        temp['answer'] = final_answer
        item['predicted_answer'] = final_answer
        ensemble_result.append(temp)
        
    # Save all ensemble results to JSON file
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(ensemble_result, f, ensure_ascii=False, indent=2) 

## We use the results:
## + True/False questions: GRPO-VI-Qwen2-7B-RAG model using zero-shot prompting.
## + Multiple-choice questions: ensemble results from three models. 
## + Free-text questions: GRPO-VI-Qwen2-7B-RAG model using few-shot prompting.

In [58]:
TF_idx = 0
MC_idx = [0, 1, 2]
FT_idx = 3
output_file = "ensemble_result_private.json"
ensemble(TF_idx, MC_idx, FT_idx, output_file)

## We use the results:
## + True/False questions: GRPO-VI-Qwen2-7B-RAG model using few-shot prompting.
## + Multiple-choice questions: ensemble results from three models. 
## + Free-text questions: GRPO-VI-Qwen2-7B-RAG model using few-shot prompting.

In [59]:
TF_idx = 3
MC_idx = [0, 1, 2]
FT_idx = 3
output_file = "ensemble_result_ver3.json"
ensemble(TF_idx, MC_idx, FT_idx, output_file)